# UNderstanding the globe 

### glob in one picture

glob finds files by a pattern, the way you would in a file manager search. The * means any text at all, so *.gguf means every file whose name ends in .gguf. The result is always a Python list of
full paths. Try it yourself in a scratch folder:


## [gguf]GPT-Generated Unified Format.

The most widely cited expansion in AI documentation and industry guides, reflecting its primary use for running large language models (LLMs)

In [11]:
import glob
print(glob.glob('data/*.gguf'))

[]


In [33]:
from gguf import GGUFReader
import glob, os
# 

path = glob.glob(os.path.expanduser('~/llama.cpp/models/*.gguf'))[0]
print(path)

r=GGUFReader(path)

# This opens the file at path and reads only the header, 
# the metadata and the tensor list. The weights stay on disk, which is why it is fast. After this line, r.fields holds the met
# adata and r.tensors holds the list of weights.

/home/gita/llama.cpp/models/ggml-vocab-bert-bge.gguf


In [31]:
r.fields.items()

odict_items([('GGUF.version', ReaderField(offset=4, name='GGUF.version', parts=[memmap([3], dtype=uint32)], data=[0], types=[<GGUFValueType.UINT32: 4>])), ('GGUF.tensor_count', ReaderField(offset=8, name='GGUF.tensor_count', parts=[memmap([0], dtype=uint64)], data=[0], types=[<GGUFValueType.UINT64: 10>])), ('GGUF.kv_count', ReaderField(offset=16, name='GGUF.kv_count', parts=[memmap([20], dtype=uint64)], data=[0], types=[<GGUFValueType.UINT64: 10>])), ('general.architecture', ReaderField(offset=24, name='general.architecture', parts=[memmap([20], dtype=uint64), memmap([103, 101, 110, 101, 114,  97, 108,  46,  97, 114,  99, 104, 105,
        116, 101,  99, 116, 117, 114, 101], dtype=uint8), memmap([8], dtype=uint32), memmap([4], dtype=uint64), memmap([ 98, 101, 114, 116], dtype=uint8)], data=[4], types=[<GGUFValueType.STRING: 8>])), ('general.name', ReaderField(offset=68, name='general.name', parts=[memmap([12], dtype=uint64), memmap([103, 101, 110, 101, 114,  97, 108,  46, 110,  97, 109

In [35]:
from gguf import GGUFReader
import glob, os

# 1. collect every big .gguf under the places models usually live
roots = ['~/.cache/llama.cpp', '~/.cache/huggingface', '~/models']
files = []
for root in roots:
    files += glob.glob(os.path.expanduser(root) + '/**/*.gguf', recursive=True)
files = [f for f in files if os.path.getsize(f) > 50_000_000]

# 2. print them so you can see what exists
for i, f in enumerate(files):
    print(i, round(os.path.getsize(f) / 1e9, 2), 'GB', f)

# 3. pick one by name, ignoring upper and lower case
matches = [f for f in files if 'nemotron' in f.lower()]
print(matches)
path = matches[0]

r = GGUFReader(path)

0 2.84 GB /home/gita/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-3-Nano-4B-GGUF/snapshots/ba223d14e45525f7fae81db77ea8cabeb2fc6c25/NVIDIA-Nemotron3-Nano-4B-Q4_K_M.gguf
1 1.93 GB /home/gita/.cache/huggingface/hub/models--bartowski--Qwen2.5-3B-Instruct-GGUF/snapshots/f302c64a2269a69fb27b2f9473b362f5bb8e78d8/Qwen2.5-3B-Instruct-Q4_K_M.gguf
2 4.92 GB /home/gita/.cache/huggingface/hub/models--bartowski--DeepSeek-R1-Distill-Llama-8B-GGUF/snapshots/935d400c2ed1a29cc2a1045df25ec7998fb4dfe4/DeepSeek-R1-Distill-Llama-8B-Q4_K_M.gguf
['/home/gita/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-3-Nano-4B-GGUF/snapshots/ba223d14e45525f7fae81db77ea8cabeb2fc6c25/NVIDIA-Nemotron3-Nano-4B-Q4_K_M.gguf']


In [36]:
from gguf import GGUFReader
import glob, os

def value_of(f):
    kind = f.types[0].name
    if kind == 'ARRAY':
        return f'[list of {len(f.data)} items]'
    if kind == 'STRING':
        return bytes(f.parts[f.data[0]]).decode()[:60]
    return f.parts[f.data[0]][0]

root = os.path.expanduser('~/.cache/huggingface/hub')
files = glob.glob(root + '/**/*.gguf', recursive=True)
files = [f for f in files if os.path.getsize(f) > 50_000_000]

for path in files:
    r = GGUFReader(path)
    meta = {k: value_of(f) for k, f in r.fields.items()}
    arch = meta.get('general.architecture')
    print(os.path.basename(path))
    print('  architecture :', arch)
    for name in ['block_count', 'embedding_length', 'attention.head_count',
                 'attention.head_count_kv', 'context_length']:
        print(f'  {name:24s}:', meta.get(f'{arch}.{name}'))
    print('  tensors      :', len(r.tensors))

NVIDIA-Nemotron3-Nano-4B-Q4_K_M.gguf
  architecture : nemotron_h
  block_count             : 42
  embedding_length        : 3136
  attention.head_count    : 40
  attention.head_count_kv : [list of 42 items]
  context_length          : 1048576
  tensors      : 263
Qwen2.5-3B-Instruct-Q4_K_M.gguf
  architecture : qwen2
  block_count             : 36
  embedding_length        : 2048
  attention.head_count    : 16
  attention.head_count_kv : 2
  context_length          : 32768
  tensors      : 434
DeepSeek-R1-Distill-Llama-8B-Q4_K_M.gguf
  architecture : llama
  block_count             : 32
  embedding_length        : 4096
  attention.head_count    : 32
  attention.head_count_kv : 8
  context_length          : 131072
  tensors      : 292


In [38]:
import fnmatch
fnmatch.fnmatchcase('NVIDIA-Nemotron-Q4.gguf', '*nemetron')

False

In [40]:
'nemotron' in 'NVIDIA-Nemotron-Q4.gguf'.lower()

True

# GET THE MODELS GGUF FILES

In [47]:
from gguf import GGUFReader
import glob, os

root=os.path.expanduser('~/.cache/huggingface/hub')
# **: A double asterisk wildcard that matches zero or more subdirectories (meaning it looks in root 
# itself, plus every nested folder inside it).

# recursive=True: Tells Python to actually treat ** as a deep, multi-level folder search. Without this flag,
# ** only searches one folder level deep.
files=glob.glob(root + '/**/*.gguf', recursive=True)
files=[f for f in files if os.path.getsize(f) > 50_000_000]
files

['/home/gita/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-3-Nano-4B-GGUF/snapshots/ba223d14e45525f7fae81db77ea8cabeb2fc6c25/NVIDIA-Nemotron3-Nano-4B-Q4_K_M.gguf',
 '/home/gita/.cache/huggingface/hub/models--bartowski--Qwen2.5-3B-Instruct-GGUF/snapshots/f302c64a2269a69fb27b2f9473b362f5bb8e78d8/Qwen2.5-3B-Instruct-Q4_K_M.gguf',
 '/home/gita/.cache/huggingface/hub/models--bartowski--DeepSeek-R1-Distill-Llama-8B-GGUF/snapshots/935d400c2ed1a29cc2a1045df25ec7998fb4dfe4/DeepSeek-R1-Distill-Llama-8B-Q4_K_M.gguf']

In [56]:
# value of each file function

from gguf import GGUFReader
import glob, os

def value_of(f):
    kind=f.types[0].name
    if kind=='ARRAY':
        return f'[list of {len(f.data)} items]'
    if kind =='STRING':
        return bytes(f.parts[f.data[0]]).decode()[:60]
    return f.parts[f.data[0]][0]



for path in files:
    r=GGUFReader(path)
    meta={k:value_of(f) for k, f in r.fields.items()}
    arch=meta.get('general.architecture')
    print(os.path.basename(path))
    print(' architecture :', arch)
    # print(arch)

NVIDIA-Nemotron3-Nano-4B-Q4_K_M.gguf
 architecture : nemotron_h
Qwen2.5-3B-Instruct-Q4_K_M.gguf
 architecture : qwen2
DeepSeek-R1-Distill-Llama-8B-Q4_K_M.gguf
 architecture : llama


In [41]:
# Three models of different designs make a good comparison,
# so this prints the key numbers for all of them side by side:

In [45]:


def value_of(f):
    kind=f.type[0].name
    # to be continued



['/home/gita/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-3-Nano-4B-GGUF/snapshots/ba223d14e45525f7fae81db77ea8cabeb2fc6c25/NVIDIA-Nemotron3-Nano-4B-Q4_K_M.gguf',
 '/home/gita/.cache/huggingface/hub/models--bartowski--Qwen2.5-3B-Instruct-GGUF/snapshots/f302c64a2269a69fb27b2f9473b362f5bb8e78d8/Qwen2.5-3B-Instruct-Q4_K_M.gguf',
 '/home/gita/.cache/huggingface/hub/models--bartowski--DeepSeek-R1-Distill-Llama-8B-GGUF/snapshots/935d400c2ed1a29cc2a1045df25ec7998fb4dfe4/DeepSeek-R1-Distill-Llama-8B-Q4_K_M.gguf']